# 00 · v30 de principio a fin

Ejecuta en orden los cuadernos que recrean v30 (RMSE oficial 247,8635, subido como
`knowledgeable-alligator_v29.parquet`): un clasificador de copia de la hora programada y un regresor del rodaje,
ambos CatBoost, sobre 177 variables, con la cola de Roma recalibrada y la proyección LOBT.

Con `PERIODO` elegimos qué se predice con la misma receta:

- `"ranking"`: enero y julio de 2026 (`ranking.parquet`, 344.841 salidas), el periodo de v30.
- `"final"`: enero, febrero, junio y julio de 2026 (`final_ranking.parquet`, 670.790 salidas). Como 2026 no trae
  etiquetas, se entrena igual con 2025 y solo se rehacen las variables de 2026 y las agregaciones que dependen de
  sus vuelos (`Data/interim_final/`, `Data/features_final/`).

| Cuaderno | Qué hace | Produce |
|---|---|---|
| `01_datos` | Ficheros del reto (bucket de OpenSky, solo lectura) | `Data/raw/`, `Data/raw_final/` |
| `02_variables_base` | METAR (IEM) y las 137 variables base | `Data/metar/`, `Data/interim*/`, `Data/features*/base/` |
| `03_adsb` | Trazas de adsb.lol, 20 variables ADS-B y 4 de puerta/stand | `Data/adsb/`, `Data/features*/adsb*` |
| `04_variables_fuente` | Las 16 variables de fuente del off-block | `Data/features*/fuentes_*` |
| `05_experimentos` | Experimentos que eligieron la receta (opcional, solo ranking) | `Data/experimentos/` |
| `06_modelo` | Predice con los modelos guardados de v30 y, si se pide, reentrena | `Submissions/<NOMBRE>*/` |
| `07_envio` | Sube un fichero al bucket del equipo (el 00 no lo ejecuta) | `registro_envio.json` |

## Periodo e interruptores

Por defecto reutilizamos lo que ya existe y no descargamos nada pesado, así que v30 se recrea con los mismos datos
con los que se entrenó. Un 1 fuerza el paso. Si cambias `PERIODO`, reinicia el kernel.

In [1]:
PERIODO = "final"   # "ranking": ene + jul 2026 (344.841 salidas) · "final": ene, feb, jun y jul 2026 (670.790)
INTERRUPTORES = {
    "DESCARGAR_RETO": 0,    # 01: vuelve a descargar training_*, ranking y submitting (y los final_* con PERIODO final)
    "DESCARGAR_METAR": 0,   # 02: descarga los METAR de 2025 y 2026 de IEM (unos minutos)
    "REHACER_BASE": 0,      # 02: recalcula las tablas de variables base desde los METAR y los ficheros del reto
    "DESCARGAR_ADSB": 0,    # 03: descarga las trazas de adsb.lol (~9 GB de ADSB.exposed + ~4 GB/día de jun/jul en GitHub)
    "REHACER_ADSB": 0,      # 03: recalcula enlace traza-vuelo, medidas y posiciones de los stands
    "REHACER_FUENTES": 0,   # 04: recalcula las variables de fuente del off-block (~7 min)
    "EXPERIMENTOS": 0,      # 05: repite los experimentos de selección (solo con PERIODO ranking)
    "REENTRENAR": 0,        # 06: además de predecir con los modelos guardados de v30, reentrena desde cero (16 min)
}
NOMBRE = "v30_recreado" if PERIODO == "ranking" else "v30_final"   # carpeta de salida en Submissions (nunca "v30")

In [2]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "cuaderno.py").exists())
sys.path.insert(0, str(ROOT / "scripts"))
import os
import time

import nbformat
from jupyter_client.kernelspec import KernelSpecManager
from nbclient import NotebookClient

for k, v in INTERRUPTORES.items():
    os.environ[f"V30_{k}"] = str(int(v))
os.environ["V30_NOMBRE"] = NOMBRE
os.environ["V30_PERIODO"] = PERIODO
from cuaderno import comprobar_paquetes, elegir_periodo, existen, interruptor, run  # noqa: E402
comprobar_paquetes()                           # los cuadernos y scripts usan el Python de este kernel
elegir_periodo(PERIODO)                        # PRC_PERIODO para este kernel y los de los cuadernos que lanza
KERNEL = "prc-venv" if "prc-venv" in KernelSpecManager().find_kernel_specs() else "python3"

CADENA = ["01_datos", "02_variables_base", "03_adsb", "04_variables_fuente", "05_experimentos", "06_modelo"]
CARPETA = ROOT / "Notebooks"
EJECUTADOS = CARPETA / "ejecutados"
EJECUTADOS.mkdir(exist_ok=True)
print(f"PERIODO = {PERIODO} · NOMBRE = {NOMBRE} · kernel {KERNEL}")
for nombre in CADENA:
    if nombre == "05_experimentos" and (not INTERRUPTORES["EXPERIMENTOS"] or PERIODO == "final"):
        print("05_experimentos: omitido", "(PERIODO = final)" if PERIODO == "final" else "(EXPERIMENTOS = 0)")
        continue
    t0 = time.time()
    nb = nbformat.read(CARPETA / f"{nombre}.ipynb", as_version=4)
    try:
        NotebookClient(nb, timeout=None, kernel_name=KERNEL, resources={"metadata": {"path": str(CARPETA)}}).execute()
    finally:
        nbformat.write(nb, EJECUTADOS / f"{nombre}.ipynb")   # con sus salidas, también si falla
    print(f"{nombre}: OK en {(time.time() - t0) / 60:.1f} min -> Notebooks/ejecutados/{nombre}.ipynb", flush=True)

PERIODO = final · NOMBRE = v30_final · kernel python3


c:\Proyectos\PRC-CHALLENGE\prc-venv\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()


01_datos: OK en 0.1 min -> Notebooks/ejecutados/01_datos.ipynb
02_variables_base: OK en 1.4 min -> Notebooks/ejecutados/02_variables_base.ipynb
03_adsb: OK en 0.2 min -> Notebooks/ejecutados/03_adsb.ipynb
04_variables_fuente: OK en 0.1 min -> Notebooks/ejecutados/04_variables_fuente.ipynb
05_experimentos: omitido (PERIODO = final)
06_modelo: OK en 2.7 min -> Notebooks/ejecutados/06_modelo.ipynb


## Resultado: comparación con el v30 enviado

In [3]:
import json
print(json.dumps(json.loads((ROOT / "Submissions" / f"{NOMBRE}_modelos" / "comparacion_v30.json").read_text(encoding="utf-8")),
                 indent=1, ensure_ascii=False))

{
 "periodo": "final",
 "enviado": {
  "sha256": "0fe1cf30303b00135ba5a6f23c56e618f524a6eeea5aa0f7732fc3caf8d1bd6e",
  "media_s": 1041.6
 },
 "con_modelos_guardados": {
  "sha256": "03faee3d822ee45b38f264f8186b47f8abc514c15c2025dceac77e29b1cdd6db",
  "identico_al_enviado": false,
  "filas": 670790,
  "filas_comunes_con_v30": 344841,
  "predicciones_iguales_%": 4.09,
  "dif_media_s": -2.56,
  "dif_abs_mediana_s": 7.0,
  "dif_abs_p99_s": 66.0,
  "rmse_entre_ambos_s": 18.97,
  "media_s": 1014.0,
  "valido": true
 }
}
